In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Đường dẫn đến thư mục dữ liệu thô (tự động nhận diện local hoặc Google Colab)
RAW = Path("RAW_DATA/raw") if Path("RAW_DATA/raw").exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/raw")

FILES = [
    "Fact", "Calendar", "Business Line", "Group", "Subgroup",
    "Channel", "Region", "Headlines", "KPI", "Ratios"
]

# Đọc toàn bộ bảng dữ liệu dưới dạng chuỗi
tables = {
    f: pd.read_csv(RAW / f"{f}.csv", dtype=str, encoding="utf-8-sig")
    for f in FILES
}

fact = tables["Fact"]

# Kiểm tra kích thước các bảng
{name: df.shape for name, df in tables.items()}

{'Fact': (8726, 10),
 'Calendar': (360, 7),
 'Business Line': (3, 2),
 'Group': (4, 3),
 'Subgroup': (15, 3),
 'Channel': (3, 2),
 'Region': (3, 2),
 'Headlines': (7, 4),
 'KPI': (5, 3),
 'Ratios': (5, 2)}

# 2. Data Preprocessing

## 2.1. Data cleaning

### Bước 1 & Bước 2: Xử lý bản ghi trùng lặp và Chuẩn hóa định dạng

In [2]:
# Định nghĩa hàm chuẩn hóa số Amount
def parse_amount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    neg = s.startswith("(") or s.startswith("-")
    s = re.sub(r"USD|\$|[()\s-]", "", s, flags=re.I)
    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+,\d{2}|\d+,\d{2}", s):   # kiểu VN: 6.720,07
        s = s.replace(".", "").replace(",", ".")
    else:                                                          # kiểu quốc tế: 1,632.79
        s = s.replace(",", "")
    return (-1 if neg else 1) * float(s)

# Tạo bản sao, giữ nguyên dữ liệu gốc
df = fact.copy()

# Đổi tên cột cho dễ thao tác
df = df.rename(columns={
    "Amount, $": "Amount",
    "Business Line ID": "BL_ID",
    "Business Line Name": "BL_Name",
    "Subgroup ID": "Subgroup_ID",
    "Group ID": "Group_ID",
    "Revenue or expense": "RevExp"
})

# Bảng chuẩn dùng để sửa khóa
SG = tables["Subgroup"]
BL = tables["Business Line"].copy()
n_goc = len(df)

# 1. Xóa dòng trùng (Thành viên: Trần Nhựt Hào)
df = df.drop_duplicates().reset_index(drop=True)

# 2. Chuẩn hoá định dạng (Thành viên: Trần Nhựt Hào)
# 2.1 Date:
df["Date"] = pd.to_datetime(
    df["Date"], format="%Y-%m-%d", errors="coerce"
).fillna(
    pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%Y%m%d", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%b %d, %Y", errors="coerce")
)

# 2.2 Amount: đưa về số thực
df["Amount"] = df["Amount"].map(parse_amount)

# 2.3 Chuẩn hoá nhãn phân loại
df["Channel"] = df["Channel"].str.strip().str.lower().map({
    "online": "Online", "store": "Store", "distributor": "Distributor"
})

df["Region"] = (
    df["Region"].str.strip().str.lower()
    .str.replace("miền ", "", regex=False)
    .map({"bắc": "Bắc", "bac": "Bắc", "trung": "Trung", "nam": "Nam"})
)

df["RevExp"] = (
    df["RevExp"].str.strip().str.lower()
    .map({
        "revenue": "Revenue", "rev": "Revenue",
        "expense": "Expense", "expenses": "Expense", "exp": "Expense"
    })
)

In [3]:
# Kiểm tra kết quả sau khi Xử lý trùng lặp và Chuẩn hóa định dạng
print(f"Số dòng ban đầu: {n_goc} dòng")
print(f"Số dòng sau xóa trùng: {len(df)} dòng (Đã giảm: {n_goc - len(df)} dòng trùng lặp)")
print("Số dòng trùng lặp hoàn toàn còn lại:", df.duplicated().sum())
print("\nThông tin dữ liệu sau chuẩn hóa định dạng:")
print(df.info())
display(df.head(10))

Số dòng ban đầu: 8726 dòng
Số dòng sau xóa trùng: 8640 dòng (Đã giảm: 86 dòng trùng lặp)
Số dòng trùng lặp hoàn toàn còn lại: 0

Thông tin dữ liệu sau chuẩn hóa định dạng:
<class 'pandas.DataFrame'>
RangeIndex: 8640 entries, 0 to 8639
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   TransactionID  8640 non-null   str           
 1   Date           8541 non-null   datetime64[us]
 2   Amount         8392 non-null   float64       
 3   BL_ID          8640 non-null   str           
 4   BL_Name        8640 non-null   str           
 5   Subgroup_ID    8640 non-null   str           
 6   Group_ID       8640 non-null   str           
 7   RevExp         8640 non-null   str           
 8   Channel        7827 non-null   str           
 9   Region         8190 non-null   str           
dtypes: datetime64[us](1), float64(1), str(8)
memory usage: 675.1 KB
None


,TransactionID,Date,Amount,BL_ID,BL_Name,Subgroup_ID,Group_ID,RevExp,Channel,Region
0,T00001,2023-01-01,2000.34,2,Sports equipment,1,1,Revenue,Store,Bắc
1,T00002,2023-01-01,1632.79,1,Nutrition and Food Supplements,1,1,Revenue,NaN,Bắc
2,T00003,2023-01-01,8792.88,2,Sports equipment,1,1,Revenue,Store,Nam
3,T00004,2023-01-01,-6978.85,1,Nutrition and Food Supplements,4,2,Expense,NaN,Bắc
4,T00005,2023-01-01,-1011.47,1,Nutrition and Food Supplements,5,2,Expense,Store,Nam
5,T00006,2023-01-01,-18503.05,2,Sports equipment,14,3,Expense,NaN,Trung
6,T00007,2023-01-01,-3786.31,1,Nutrition and Food Supplements,8,2,Expense,Online,Nam
7,T00008,2023-01-01,9812.00,3,Sportswear,1,1,Revenue,Store,Bắc
8,T00009,2023-01-01,-1932.54,2,Sports equipment,10,3,Expense,Distributor,Trung
9,T00010,2023-01-01,2520.50,1,Nutrition and Food Supplements,1,1,Revenue,Store,Nam
